# 04_improve_imbalance — Role D: Improvement 2 (Class Imbalance)

Mục tiêu: kiểm tra liệu hai cách xử lý mất cân bằng lớp có cải thiện recall của các lớp `Dropout` và `Enrolled` hay không. Toàn bộ mô hình dùng split chung từ `src.data.get_train_test()`; chỉ M2b được SMOTE trên **tập train sau khi split**.

## 1. Import và xác định repo root

Cell này chỉ nhập các helper dùng chung. Notebook không tự cài đặt lại logic load/split dữ liệu hay tính metrics.

In [1]:
%matplotlib inline

import sys
from pathlib import Path

import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier


def find_repo_root(start: Path | None = None) -> Path:
    """Return the nearest ancestor containing the shared data pipeline."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy repo root chứa src/data.py")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import get_train_test
from src.evaluate import evaluate_model
from src.visualize import plot_tree_figure

FIGURES_DIR = REPO_ROOT / "figures"
OUTPUTS_DIR = REPO_ROOT / "outputs"
RESULTS_PATH = OUTPUTS_DIR / "results.csv"
CLASS_ORDER = ["Dropout", "Enrolled", "Graduate"]

print(f"Repo root: {REPO_ROOT.name}")

Repo root: Lab2_DecisionTree


## 2. Lấy split chung một lần

Đây là lần gọi duy nhất tới `get_train_test()`. M2a và M2b dùng cùng tập test để so sánh công bằng với M0.

In [2]:
X_train, X_test, y_train, y_test = get_train_test()

assert set(y_train.unique()) == set(CLASS_ORDER)
assert set(y_test.unique()) == set(CLASS_ORDER)

split_summary = pd.DataFrame({
    "train_n": y_train.value_counts().reindex(CLASS_ORDER),
    "test_n": y_test.value_counts().reindex(CLASS_ORDER),
})
display(split_summary)
print(f"X_train: {X_train.shape}; X_test: {X_test.shape}")

,train_n,test_n
Target,,
Dropout,1137,284
Enrolled,635,159
Graduate,1767,442


X_train: (3539, 90); X_test: (885, 90)


## 3. M2a — Cân bằng bằng trọng số lớp

`class_weight='balanced'` tăng trọng số lỗi cho các lớp ít mẫu khi xây cây. Dữ liệu đầu vào không bị resample.

In [3]:
m2a = DecisionTreeClassifier(class_weight="balanced", random_state=42)
m2a.fit(X_train, y_train)

m2a_result = evaluate_model(
    m2a, X_train, y_train, X_test, y_test,
    model_id="M2a",
    model_name="Decision Tree with Balanced Class Weights",
    params={"class_weight": "balanced", "random_state": 42},
    author="D",
    classification_report_path=OUTPUTS_DIR / "classification_report_M2a.txt",
    confusion_matrix_path=FIGURES_DIR / "D_cm_M2a.png",
)

m2a_tree_path = plot_tree_figure(
    m2a, X_train.columns, m2a.classes_,
    FIGURES_DIR / "D_tree_M2a.png",
    max_depth=4, figsize=(30, 18), dpi=200, fontsize=7,
    title="M2a Decision Tree — Balanced Class Weights (first 4 levels)",
)

display(pd.DataFrame([m2a_result]))
print(f"Saved tree: {m2a_tree_path.name}")

,model_id,model_name,params,train_acc,test_acc,error_rate,precision_macro,recall_macro,f1_macro,roc_auc_macro,recall_dropout,recall_enrolled,recall_graduate,tree_depth,n_leaves,author
0,M2a,Decision Tree with Balanced Class Weights,"{""class_weight"":""balanced"",""random_state"":42}",1.0,0.650847,0.349153,0.58425,0.587848,0.585409,0.705321,0.679577,0.339623,0.744344,28,696,D


Saved tree: D_tree_M2a.png


In [4]:
m2a_tree_full_path = plot_tree_figure(
    m2a, X_train.columns, m2a.classes_,
    FIGURES_DIR / "D_tree_M2a_full.png",
    max_depth=None, figsize=(60, 30), dpi=150,
    title="M2a Decision Tree — Balanced Class Weights (full tree)",
)

print(f"Saved full tree: {m2a_tree_full_path.name}")

Saved full tree: D_tree_M2a_full.png


## 4. M2b — SMOTE chỉ trên tập train

`SMOTE.fit_resample()` được gọi sau khi `get_train_test()` đã chia dữ liệu. Tập test gốc không bị thay đổi, nên không có rò rỉ dữ liệu từ các điểm tổng hợp vào đánh giá.

In [5]:
train_counts_before_smote = y_train.value_counts().reindex(CLASS_ORDER).copy()
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
train_counts_after_smote = y_train_smote.value_counts().reindex(CLASS_ORDER)

# Guardrails: SMOTE changes only a new training matrix; original train/test stay intact.
assert y_train.value_counts().reindex(CLASS_ORDER).equals(train_counts_before_smote)
assert X_test.shape[0] == y_test.shape[0] == 885
assert train_counts_after_smote.nunique() == 1

smote_summary = pd.DataFrame({
    "before_SMOTE_train": train_counts_before_smote,
    "after_SMOTE_train": train_counts_after_smote,
})
display(smote_summary)
print(f"Resampled train shape: {X_train_smote.shape}")

,before_SMOTE_train,after_SMOTE_train
Target,,
Dropout,1137,1767
Enrolled,635,1767
Graduate,1767,1767


Resampled train shape: (5301, 90)


In [6]:
m2b = DecisionTreeClassifier(random_state=42)
m2b.fit(X_train_smote, y_train_smote)

# Evaluate on the original split, not on SMOTE-generated samples.
m2b_result = evaluate_model(
    m2b, X_train, y_train, X_test, y_test,
    model_id="M2b",
    model_name="Decision Tree with SMOTE on Training Set",
    params={"sampler": "SMOTE", "smote_random_state": 42, "random_state": 42},
    author="D",
    classification_report_path=OUTPUTS_DIR / "classification_report_M2b.txt",
    confusion_matrix_path=FIGURES_DIR / "D_cm_M2b.png",
)

display(pd.DataFrame([m2b_result]))

,model_id,model_name,params,train_acc,test_acc,error_rate,precision_macro,recall_macro,f1_macro,roc_auc_macro,recall_dropout,recall_enrolled,recall_graduate,tree_depth,n_leaves,author
0,M2b,Decision Tree with SMOTE on Training Set,"{""random_state"":42,""sampler"":""SMOTE"",""smote_ra...",1.0,0.688136,0.311864,0.636513,0.642937,0.638747,0.742122,0.707746,0.465409,0.755656,39,847,D


## 5. So sánh recall từng lớp: M0, M2a, M2b

Đây là tiêu chí chính của cải tiến xử lý mất cân bằng. Nếu test accuracy tổng của M2a hoặc M2b giảm so với M0, đó là kết quả bình thường và có thể được mong đợi: mô hình đang đánh đổi một phần độ chính xác trên lớp đa số để nhận diện tốt hơn các lớp thiểu số, đặc biệt `Enrolled`. Không tối ưu lại chỉ nhằm đưa accuracy trở về bằng M0; cần đánh giá đánh đổi bằng recall của từng lớp.

In [7]:
results = pd.read_csv(RESULTS_PATH)
required_rows = {"M0", "M2a", "M2b"}
assert required_rows.issubset(set(results["model_id"])), "Thiếu kết quả M0/M2a/M2b trong results.csv"

m0_rows = results.loc[(results["model_id"] == "M0") & (results["author"] == "B")]
assert len(m0_rows) == 1, "Cần đúng một dòng M0 do Role B tạo"

comparison = (
    results.loc[results["model_id"].isin(["M0", "M2a", "M2b"])]
    .set_index("model_id")
    .loc[["M0", "M2a", "M2b"], [
        "model_name", "test_acc", "recall_dropout",
        "recall_enrolled", "recall_graduate",
    ]]
    .rename(columns={
        "test_acc": "test_accuracy",
        "recall_dropout": "recall_Dropout",
        "recall_enrolled": "recall_Enrolled",
        "recall_graduate": "recall_Graduate",
    })
)

recall_deltas_vs_m0 = comparison.loc[["M2a", "M2b"], [
    "recall_Dropout", "recall_Enrolled", "recall_Graduate"
]].subtract(comparison.loc["M0", [
    "recall_Dropout", "recall_Enrolled", "recall_Graduate"
]], axis=1)

display(comparison.style.format({
    "test_accuracy": "{:.4f}",
    "recall_Dropout": "{:.4f}",
    "recall_Enrolled": "{:.4f}",
    "recall_Graduate": "{:.4f}",
}))
display(recall_deltas_vs_m0.style.format("{:+.4f}").set_caption("Recall change versus M0"))

,model_name,test_accuracy,recall_Dropout,recall_Enrolled,recall_Graduate
model_id,,,,,
M0,Baseline Decision Tree (Gini),0.6689,0.6796,0.3836,0.7647
M2a,Decision Tree with Balanced Class Weights,0.6508,0.6796,0.3396,0.7443
M2b,Decision Tree with SMOTE on Training Set,0.6881,0.7077,0.4654,0.7557


,recall_Dropout,recall_Enrolled,recall_Graduate
model_id,,,
M2a,+0.0000,-0.0440,-0.0204
M2b,+0.0282,+0.0818,-0.0090


## 6. Kết luận

Dùng bảng recall phía trên cùng hai confusion matrices `D_cm_M2a.png` và `D_cm_M2b.png` để viết phần báo cáo. Kết luận phải nêu đồng thời các cải thiện/giảm sút recall theo từng lớp và thay đổi accuracy tổng, thay vì chỉ chọn model có accuracy cao nhất.